## Filesystem Hierarchy Standard (FHS) — Grouped by Purpose

Instead of memorizing 15 directories as a flat list, group them by **why they exist**. Every directory hanging off `/` (the root of the entire system) answers one of these questions:

| Purpose | Question it answers | Directories |
|---|---|---|
| **Programs** | What runs on this system? | `/bin`, `/sbin`, `/usr` |
| **Shared libraries** | What code do programs borrow? | `/lib` |
| **Configuration** | How is everything set up? | `/etc` |
| **User data** | Whose files are these? | `/home`, `/root` |
| **Changing / runtime data** | What's happening or accumulating right now? | `/var`, `/tmp`, `/run` |
| **Live kernel info** | What does the kernel know right this second? | `/proc`, `/sys` |
| **Hardware** | How do I talk to devices? | `/dev` |
| **Boot** | How does the system start? | `/boot` |
| **External storage** | What's plugged in or mounted? | `/mnt`, `/media` |

```shell
                                       /
                                       │
     ┌───────────┬───────────┬─────────┼─────────┬───────────┬─────────────┐
     │           │           │         │         │           │             │
 PROGRAMS     CONFIG     USER DATA  RUNTIME/LIVE  HARDWARE     BOOT      EXTERNAL
 what runs    settings   your stuff state/kernel  devices    startup     storage
     │           │           │         │         │           │             │
  /bin ─┐       /etc       /home     /var        /dev       /boot        /mnt
  /sbin ┼─/usr             /root     /tmp                                /media
  /lib ─┘                            /run
                                     /proc *
                                     /sys  *

                    * virtual — built live by the kernel, not stored on disk
```

You'll notice `/bin`, `/sbin` and `/lib` all route through `/usr` on modern Ubuntu — that's the "usr-merge," explained in the Programs section below. The sections that follow walk through each purpose group in turn.

### 🧰 Programs — `/bin`, `/sbin`, `/usr`

**`/bin`** (binaries): essential command-line programs Linux needs to function — `ls`, `cp`, `mv`, `cat`, `mkdir`. The toolbox Linux itself relies on.

**`/sbin`** (system binaries): administrative tools for managing the OS — networking (`ip`, `ifconfig`, `route`), disk management (`fdisk`, `mkfs`, `parted`), booting (`grub-install`, `init`), recovery (`fsck`, `e2fsck`). Regular users rarely touch these day-to-day; admins do.

**`/usr`** (Unix System Resources): the main software library of the machine — most installed applications, their binaries, libraries, and docs live here.

```bash
$ ls /usr/bin | wc -l
809
```

**The modern twist (usr-merge):** on older Unix systems, `/bin` held only what you needed *before* `/usr` was mounted, during early boot. Today `/usr` is always available at boot, so that split is obsolete — but plenty of scripts still hardcode `/bin/bash` or `/sbin/ip`. Rather than break them, modern distros (Ubuntu included) turned `/bin` and `/sbin` into **symlinks** into `/usr`:

```bash
$ ls -la / | grep -E '^l'
lrwxrwxrwx  1 root root  7 Apr 22  2024 bin -> usr/bin
lrwxrwxrwx  1 root root  7 Apr 22  2024 lib -> usr/lib
lrwxrwxrwx  1 root root  9 Apr 22  2024 lib64 -> usr/lib64
lrwxrwxrwx  1 root root  8 Apr 22  2024 sbin -> usr/sbin
```

So `/bin/ls` and `/usr/bin/ls` are literally the same file — old scripts keep working, and there's really only one location (`/usr`) to maintain.

**Support relevance:** if you ever see "duplicate binary" confusion — `which ls` reports `/bin/ls`, some doc says `/usr/bin/ls` — they're not duplicates, it's one file via symlink. Don't waste time hunting for "two versions."

### 🧩 Shared Libraries — `/lib`

Programs need shared code. Instead of every program bundling the same code, Linux stores common functionality once, in `/lib`, and programs borrow it at runtime. Saves disk space, and a single library update patches every program that depends on it. Think of it like one `requirements.txt`-installed package that many of your Python scripts `import` from, instead of each script vendoring its own copy.

Same usr-merge story applies here: `/lib` is a symlink to `/usr/lib` (see the previous section). `/lib64` similarly points to `/usr/lib64` (64-bit libraries).

**GOTCHA:** "missing shared library" errors — `error while loading shared libraries: libfoo.so.2: cannot open shared object file` — are a `/lib` (really `/usr/lib`) problem. The program itself exists; a dependency it needs *at runtime* doesn't. It's the Linux equivalent of a missing pip package, except the "package" is a compiled `.so` file the whole system shares.

### ⚙️ Configuration — `/etc`

The settings folder for the entire operating system. Almost every application has a config file somewhere under `/etc`, rather than bundled inside the application itself. Network settings, SSH config (`/etc/ssh/sshd_config`), DNS (`/etc/resolv.conf`), cron (`/etc/crontab`) — all here.

**Developer bridge:** this is the system-wide equivalent of a `.env` or `settings.py` — except instead of one file per project, it's one *directory tree* per machine, shared by everything installed on it.

**TICKET SCENARIO:** "The web server won't start after a config change." Your first stop is almost always `/etc/<service>/` to see what was edited, then the service's logs in `/var/log` to see *why* it rejected the new config.

### 👤 User Data — `/home`, `/root`

**`/home`**: personal workspace for every regular user account — documents, downloads, projects. Each user gets their own subdirectory (`/home/dennie`) so files stay separated between accounts on the same machine.

**`/root`**: **NOT** the root of the filesystem — that's `/`. This is specifically the home directory of the **root user** (the administrator account). It's just one more folder living inside `/`, kept separate from `/home` for security reasons — root's home shouldn't sit somewhere regular users can browse into.

**GOTCHA:** confusing `/` and `/root` is one of the most common beginner mix-ups. `cd /root` and `cd /` land you in completely different places, and only root (or someone with sudo) can even read into `/root` by default.

### 🔄 Changing & Runtime Data — `/var`, `/tmp`, `/run`

These three all hold data that changes constantly, but each has a different *lifetime*:

| Directory | Survives reboot? | Typical contents |
|---|---|---|
| `/var` | ✅ yes, persists on disk | logs, mail queues, package caches, databases |
| `/tmp` | ⚠️ usually wiped on reboot | scratch files apps create while running |
| `/run` | ❌ always wiped on reboot | PID files, sockets, lock files — current-boot state only |

**`/var`** (variable): files that constantly grow — `/var/log` (system logs, your #1 troubleshooting destination), `/var/mail`, `/var/lib` (app databases/state), `/var/cache`. When troubleshooting anything, `/var/log` is one of the first places to check.

```bash
$ tail /var/log/syslog
2026-08-22T11:17:01.883694+00:00 HP CRON[29708]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
```

**`/tmp`**: scratch space for running applications. Many systems clear it on reboot. Don't store anything important here.

**`/run`** — the piece missing from the original notes. Holds runtime data for the **current boot only**: PID files, sockets, lock files. It's actually `tmpfs` — a RAM-backed filesystem, not real disk:

```bash
$ df -h /run
Filesystem   Size  Used Avail Use% Mounted on
none         3.9G  468K  3.9G   1% /run

$ cat /run/crond.pid
156
```

Older docs and scripts reference `/var/run` — on modern systems that's now just a symlink to `/run`:

```bash
$ ls -la /var/run
lrwxrwxrwx 1 root root 4 Feb 10  2026 /var/run -> /run
```

**Why the split matters for support:** if a service's PID file (`/run/foo.pid`) is stale after a crash, a reboot clears it automatically. A corrupted or exploding log in `/var/log`, on the other hand, survives reboots and keeps growing until *you* rotate or clear it. Same "changing data" category, very different failure modes.

### 🖥️ Live Kernel Info — `/proc`, `/sys`

Neither of these exists on your hard disk — the kernel builds them live, in memory, the moment you read them.

**`/proc`**: a live dashboard of running processes and kernel state — CPU, memory, per-process info. Monitoring tools (`top`, `ps`) read from here under the hood.

```bash
$ cat /proc/meminfo
MemTotal:        8055396 kB
MemFree:         5030064 kB
MemAvailable:    6409356 kB
Buffers:           29944 kB
Cached:          1466988 kB
```

**`/sys`** (sysfs) — the other piece missing from the original notes. A structured, hierarchical view of every device and driver the kernel knows about. Where `/dev` gives you a *file to talk to* a device, `/sys` gives you that device's *attributes and controls* — power state, driver bindings, tunable parameters. This is also the interface `udev` rules act on when a device is plugged in.

```bash
$ ls /sys/class/net
eth0  lo

$ cat /sys/class/net/eth0/operstate
up
```

**Developer bridge:** think of `/proc` as a live `SELECT * FROM processes` and `/sys` as a live `SELECT * FROM devices` — except instead of SQL, you just `cat` a path.

**TICKET SCENARIO:** "Why is this network interface down?" — `ip link` gives you the summary; `cat /sys/class/net/eth0/operstate` gives you the raw kernel-reported state behind that summary.

### 🔌 Hardware — `/dev`

In Linux, almost everything is treated as a file — hard drives, USB devices, keyboards, mice, terminals all appear as files inside `/dev`.

```bash
$ ls -l /dev/sda
brw-rw---- 1 root disk 8, 0 Aug 21 21:48 /dev/sda
```

One file-based interface for hardware and regular files alike — programs can read/write a disk or a terminal the same way they read/write any file. That uniformity is a big part of why Linux is so scriptable.

### 🥾 Boot — `/boot`

Everything needed to start Linux: the kernel image, boot loader files (GRUB), initial ramdisk. Without these, the OS can't boot at all.

### 💾 External Storage — `/mnt`, `/media`

**`/media`**: where auto-mounted removable devices show up — USB drives, DVDs — usually handled automatically by the desktop environment or udev.

**`/mnt`**: where admins *manually* mount disks, network shares, or other storage. Unlike Windows' drive letters (`D:\`), Linux attaches new storage somewhere inside the existing tree — there's no separate namespace, just another path. This is exactly how WSL exposes your Windows drives: `/mnt/c`, `/mnt/d`.

---

### On the video comment: "still a lot more, you didn't even mention `/run` or `/sys`..."

Fair callout. Two things from it were genuinely missing and got folded into the notes above: **`/run`** and **`/sys`**, plus the `/bin`/`/sbin`/`/lib` → `/usr` symlink detail (the "usr-merge").

The rest of the comment — nftables firewall rules, udev rule authoring, bash's ~7000-line man page — is all true, but it's a different layer of the system (networking/firewall, device management, shell scripting), not filesystem *structure*. Those get their own notes when we reach Pillar 6 (networking) and Pillar 9 (shell scripting) — no reason to cram everything into one document just because a comment mentioned it.

---

The notebook is restructured. Here's what changed:

Grouped by purpose (per your suggestion): Programs (/bin,/sbin,/usr) → Shared Libraries (/lib) → Configuration (/etc) → User Data (/home,/root) → Changing/Runtime Data (/var,/tmp,/run) → Live Kernel Info (/proc,/sys) → Hardware (/dev) → Boot (/boot) → External Storage (/mnt,/media) — each as its own cell, opening with an intro cell that has a summary table + an ASCII tree diagram.

From the comment, I added:

/run — tmpfs, boot-scoped runtime state (PID files, sockets), with a real df -h /run and cat /run/crond.pid from your system, plus the /var/run → /run symlink
/sys — sysfs, the structured device/driver tree the kernel exposes (vs /dev's "just a file"), demoed with /sys/class/net/eth0/operstate
The /bin, /sbin, /lib → /usr symlink fact (usr-merge), verified live on your machine — all real output, not fabricated
What I didn't add: the nftables/udev/bash-manual complaints — that's a different topic (networking, device rules, shell scripting), not filesystem structure, so I left a closing cell explaining that call rather than silently ignoring the comment.

Everything's real output pulled from your actual WSL session, matching the style of your existing notes (bold terms, casual tone, live command+output blocks).

One thing worth flagging: I haven't gone back to the root user / privilege delegation topic from Part 1 of our chat yet — want to continue there next, or keep going on the notebook first?